In [1]:
from abs_affinity_based_slotting.config import RAW_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.demand import build_cooccurrence, JaccardAffinity, TopKFilter
from abs_affinity_based_slotting.warehouse import occupied_locations, build_location_costs, build_bay_distance_matrix
from abs_affinity_based_slotting.slotting import build_instance, slotting_cost
from abs_affinity_based_slotting.methods import DemandGreedySlotting, LocalSearchSlotting
from abs_affinity_based_slotting.evaluation import Evaluator

In [2]:
# Cell 2 — Datos
loader = WarehouseDataLoader(RAW_DIR)
ds = loader.load_all()
split = split_picking_events(ds.picking_events, test_size=0.2)

In [3]:
# Cell 3 — Afinidad
universe = occupied_locations(ds.initial_stock)["sku"].to_numpy()

co = build_cooccurrence(split.train, skus=universe)
A = JaccardAffinity().build(co.matrix, co.support, co.n_batches)
A = TopKFilter(k=10).filter(A)

print(f"SKUs: {len(universe)}, affinity nnz: {A.nnz}")

SKUs: 27000, affinity nnz: 350986


In [4]:
# Cell 4 — Instancia
from abs_affinity_based_slotting.demand import build_sku_demand

sku_demand   = build_sku_demand(split.train) # f
loc_costs    = build_location_costs(ds.initial_stock, ds.distances) # c
bay_distance = build_bay_distance_matrix(ds.distances) # D

instance = build_instance(
    sku_demand, loc_costs, bay_distance,
    initial_stock=ds.initial_stock,
    skus=universe,
    affinity=A,
)
print(instance)


SlottingInstance(n_skus=27000, n_locations=30000, n_bays=1001, affinity_edges=350986)


In [5]:
evaluator = Evaluator.from_tables(ds.coordinates, ds.distances, ds.initial_stock)

lam = 0.5
method = LocalSearchSlotting(seed_method=DemandGreedySlotting(), lam=lam)
assignment = method.solve(instance)

metrics = evaluator.evaluate(assignment, split.test)
print(f"lam={lam}  mean={metrics.mean_batch_distance:.0f}  p95={metrics.p95_batch_distance:.0f}")

lam=0.5  mean=26964  p95=32759
